In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Preprocessing for jsonl file

In [ ]:
from datasets import load_dataset
dataset = load_dataset("json", data_files="QA_conv4_rewrite.json")

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
# def format_example(example):
#     messages = example["messages"]

#     system = ""
#     user = ""
#     assistant = ""

#     for msg in messages:
#         if msg["role"] == "system":
#             system = msg["content"]
#         elif msg["role"] == "user":
#             user = msg["content"]
#         elif msg["role"] == "assistant":
#             assistant = msg["content"]

#     prompt = f"<s>[INST] {system}\n\n{user} [/INST] {assistant} </s>"

#     return {"text": prompt}

In [ ]:
# dataset = dataset.map(format_example)

Map:   0%|          | 0/727 [00:00<?, ? examples/s]

# Tool Calls Fine-tuning

ref: Unsloth

    @software{unsloth,
    author = {Daniel Han, Michael Han and Unsloth team},
    title = {Unsloth},
    url = {https://github.com/unslothai/unsloth},
    year = {2023}
    }

## Install

In [6]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
# pip install --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## Unsloth


In [7]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Llama-3.1-70B-bnb-4bit",
    "unsloth/Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.2: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


AttributeError: module 'huggingface_hub.constants' has no attribute 'HF_HUB_ENABLE_HF_TRANSFER'

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128  ##
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,  ##
    lora_dropout = 0.05, # Supports any, but = 0 is optimized  ##
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

### Data Prep

In [ ]:
from unsloth.chat_templates import get_chat_template
# from llama_api import TOOLS

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

# add tools definition if required
def formatting_prompts_func(examples):
    convos = examples["messages"]
    # texts = [tokenizer.apply_chat_template(convo, tools = TOOLS, tokenize = False, add_generation_prompt = False) for convo in convos]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }


In [ ]:
from unsloth.chat_templates import standardize_sharegpt
# dataset = standardize_sharegpt(dataset['train'], aliases_for_system=["system"], aliases_for_user=["user"], aliases_for_assistant=["assistant"])
# my data is in standard format, doesn't need standardize_sharegpt，so map directly
dataset = dataset['train'].map(formatting_prompts_func, batched = True,)
# dataset = dataset.map(formatting_prompts_func, batched = True,)

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,  ##
        warmup_steps = 10,  ##
        num_train_epochs = 2, # Set this for 1 full training run.
        # max_steps = 279,  ## 279 = 93 * 3 = 3 epochs
        learning_rate = 2e-4,  ##
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.05,  ##
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

We also use Unsloth's train_on_completions method to only train on the assistant outputs and ignore the loss on the user's inputs.

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

We verify masking is actually done:

In [ ]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

training results

In [ ]:
trainer_stats = trainer.train()

<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!



We use `min_p = 0.1` and `temperature = 1.5`. Read this [Tweet](https://x.com/menhguin/status/1826132708508213629) for more information on why.

In [ ]:
# from unsloth.chat_templates import get_chat_template

# tokenizer = get_chat_template(
#     tokenizer,
#     chat_template = "llama-3.1",
# )
# FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# messages = [
#       {
#         "role": "system",
#         "content": "You are an economic data assistant with access to FRED API. Available FRED Economic Indicators:\n\nFormat: - SERIES_ID - Indicator Name - Unit\n- TREAST - US Treasuries Held by the Fed - Billions of Dollars\n- WSHOMCB - Mortgage Backed Sec Held by the Fed - Millions of Dollars\n- WALCL - All Fed Reserve Banks - Total Assets - Millions of Dollars\n- TOTBKCR - Bank Credit of All Commercial Banks - Billions of Dollars\n- TOTALSEC - Securitized Total Consumer Loans - Millions of Dollars\n- TOTALSL - Total Consumer Credit Outstanding - Millions of Dollars\n- INVEST - Total Investments All Commercial Banks - Millions of Dollars\n- USGSEC - US Treasury and Agency Securities at All Commercial Banks - Billions of Dollars\n- CONSUMER - Total Consumer Loans - Millions of Dollars\n- BUSLOANS - Total Commercial/Industrial Loans - Billions of Dollars\n- DALLCACBEP - Delinquencies On All Loans And Leases - Billions of Dollars\n- T10Y2Y - US 10-YR / 2-YR Spread - Percent\n- TB3MS - 3-Month T-Bill: Secondary Market Rate - Percent\n- DGS10 - 10-Yr Treasury Const. Maturity Rate - Percent\n- GFDEBTN - Federal Government Debt (Public) - Millions of Dollars\n- FYOINT - Interest on National Debt - Millions of Dollars\n- FYONET - Federal Spending - Millions of Dollars\n- FYFR - Federal Receipts - Millions of Dollars\n- FYFSD - Budget Deficit/Surplus - Millions of Dollars\n- CDSP - Consumer Debt/Income Ratio - Percent\n- PERMIT - New Home Permits - Thousands of Units\n- HSN1F - New Home Sales - Thousands\n- CMDEBT - Outstanding Mortgage Debt - Millions of Dollars\n- DGORDER - Manufacturers' New Orders - Millions of Dollars\n- TCU - Capacity Utilization: Total Industry - Percent\n- TTLCONS - Total Construction Spending - Millions of Dollars\n- BUSINV - Total Business Inventories - Millions of Dollars\n- ALTSALES - Light Weight Vehicle Sales - Millions of Units\n- UMCSENT - Univ of Michigan: Consumer Sentiment - Index 1966:Q1=100\n- STLFSI - St. Louis Financial Stress Index - Index\n- WTISPLC - Spot Oil Price - West Texas Intermediate - Dollars per Barrel\n- CPIAUCSL - Consumer Price Index: Seasonally Adj. - Index 1982-1984=100\n- UNRATE - Civilian Total Unemployment Rate - Percent\n- UEMP27OV - Long Term Unemployment: >27 WKS - Thousands of Persons\n- UEMPMED - Length of Unemployment - Weeks\n- CE16OV - Total US Workforce - Thousands of Persons\n- EMRATIO - US Employment/Population Ratio - Percent\n- POP - US Population - Thousands\n- AHEMAN - Avg Hourly Earnings: Manufacturing - Dollars per Hour\n- AWHMAN - Avg Weekly Hours: Manufacturing - Hours\n- AWOTMAN - Avg Weekly OT Hours: Manufacturing - Hours\n- DEXUSUK - USD/GBP Currency Exchange Rate - U.S. Dollars to One U.K. Pound Sterling\n- DEXUSEU - USD/EUR Currency Exchange Rate - U.S. Dollars to One Euro\n- DEXJPUS - JPN/USD Currency Exchange Rate - Japanese Yen to One U.S. Dollar\n- DEXMXUS - MXP/USD Currency Exchange Rate - Mexican Pesos to One U.S. Dollar\n- DEXCAUS - CAD/USD Currency Exchange Rate - Canadian Dollars to One U.S. Dollar\n- DEXCHUS - CNY/USD Currency Exchange Rate - Chinese Yuan Renminbi to One U.S. Dollar\n- COMPOUT - Commercial Paper Outstanding - Billions of Dollars\n- VIXCLS - CBOE Volatility Index - Index\n- GDP - US Gross Domestic Product - Billions of Dollars\n- GNP - US Gross National Product - Billions of Dollars\n- GDI - US Gross Domestic Income - Billions of Dollars\n- NETFI - US Current Account Balance - Billions of Dollars\n- EXPGS - US Exports Goods & Services - Billions of Dollars\n- IMPGS - US Imports Goods & Services - Billions of Dollars\n- DGI - Fed Govt: Defense Budget - Billions of Dollars\n- FGRECPT - Fed Govt: Tax Receipts - Billions of Dollars\n- TGDEF - Fed Govt: Budget Deficit - Billions of Dollars\n- CP - Corporate Profits After Tax - Billions of Dollars\n- DIVIDEND - Corporate Dividends - Billions of Dollars\n- PI - Personal Income - Billions of Dollars\n- PSAVE - Personal Savings - Billions of Dollars\n- PCE - Personal Consumption Expenditures - Billions of Dollars\n- PSAVERT - Personal Savings Rate - Percent\n- MORTGAGE30US - 30-yr Conventional Mortgage Rate - Percent\n- DPCREDIT - Discount Rate - Percent\n- FEDFUNDS - Effective Federal Funds Rate - Percent\n- M1 - M1 Money Supply - Billions of Dollars\n- M2 - M2 Money Supply - Billions of Dollars\n- M1V - Velocity of M1 Money Stock - Ratio\n- M2V - Velocity of M2 Money Stock - Ratio\n- PPIACO - Producer Price Index: All Commodities - Index 1982=100\n- IMPCH - Imports from China - Millions of Dollars\n- IMPJP - Imports from Japan - Millions of Dollars\n- IMPMX - Imports from Mexico - Millions of Dollars\n- IMPCA - Imports from Canada - Millions of Dollars\n- IMPGE - Imports from Germany - Millions of Dollars\n- IMPUK - Imports from UK - Millions of Dollars\n- EXPCH - Exports to China - Millions of Dollars\n- EXPJP - Exports to Japan - Millions of Dollars\n- EXPMX - Exports to Mexico - Millions of Dollars\n- EXPCA - Exports to Canada - Millions of Dollars\n- EXPGE - Exports to Germany - Millions of Dollars\n- EXPUK - Exports to UK - Millions of Dollars\n- BOPGEXP - Exports: Goods - Millions of Dollars\n- BOPGIMP - Imports: Goods - Millions of Dollars\n- BOPGTB - Balance: Goods - Millions of Dollars\n- BOPSIMP - Imports: Services - Millions of Dollars\n- BOPSTB - Balance: Services - Millions of Dollars\n- BOPGSTB - Balance: Goods & Services - Millions of Dollars\nNote: (M)=Monthly, (Q)=Quarterly, (W)=Weekly, (D)=Daily, (Y)=Yearly\n\nWhen asked about economic data, use the get_fred_data function with the appropriate series_id.\n\nYou will receive data from ALL tool calls.\n\nIMPORTANT: \n1. Always specify start_date and end_date based on the user's question. Always use YYYY-MM-DD format, NOT relative dates like \"-2y\"\n2. If no time period specified, use recent 1 year by default\n3. Each series_id should only be called ONCE with a single continuous date range.\n   - WRONG: calling GDP twice with 2018-2020 and 2021-2023\n   - RIGHT: calling GDP once with 2018-2023\n4. Today is 2026-02-25\n"
#       },
#       {
#         "role": "user",
#         "content": "How has consumer sentiment and saving behavior changed since 2024?"
#       }]

# # add tools
# inputs = tokenizer.apply_chat_template(
#     messages,
#     # tools = TOOLS,
#     tokenize = True,
#     add_generation_prompt = True, # Must add for generation
#     return_tensors = "pt",
# ).to("cuda")

# outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True,
#                          temperature = 1.5, min_p = 0.1)
# tokenizer.batch_decode(outputs)

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
# FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# messages = [
#       {
#         "role": "system",
#         "content": "You are an economic data assistant with access to FRED API. Available FRED Economic Indicators:\n\nFormat: - SERIES_ID - Indicator Name - Unit\n- TREAST - US Treasuries Held by the Fed - Billions of Dollars\n- WSHOMCB - Mortgage Backed Sec Held by the Fed - Millions of Dollars\n- WALCL - All Fed Reserve Banks - Total Assets - Millions of Dollars\n- TOTBKCR - Bank Credit of All Commercial Banks - Billions of Dollars\n- TOTALSEC - Securitized Total Consumer Loans - Millions of Dollars\n- TOTALSL - Total Consumer Credit Outstanding - Millions of Dollars\n- INVEST - Total Investments All Commercial Banks - Millions of Dollars\n- USGSEC - US Treasury and Agency Securities at All Commercial Banks - Billions of Dollars\n- CONSUMER - Total Consumer Loans - Millions of Dollars\n- BUSLOANS - Total Commercial/Industrial Loans - Billions of Dollars\n- DALLCACBEP - Delinquencies On All Loans And Leases - Billions of Dollars\n- T10Y2Y - US 10-YR / 2-YR Spread - Percent\n- TB3MS - 3-Month T-Bill: Secondary Market Rate - Percent\n- DGS10 - 10-Yr Treasury Const. Maturity Rate - Percent\n- GFDEBTN - Federal Government Debt (Public) - Millions of Dollars\n- FYOINT - Interest on National Debt - Millions of Dollars\n- FYONET - Federal Spending - Millions of Dollars\n- FYFR - Federal Receipts - Millions of Dollars\n- FYFSD - Budget Deficit/Surplus - Millions of Dollars\n- CDSP - Consumer Debt/Income Ratio - Percent\n- PERMIT - New Home Permits - Thousands of Units\n- HSN1F - New Home Sales - Thousands\n- CMDEBT - Outstanding Mortgage Debt - Millions of Dollars\n- DGORDER - Manufacturers' New Orders - Millions of Dollars\n- TCU - Capacity Utilization: Total Industry - Percent\n- TTLCONS - Total Construction Spending - Millions of Dollars\n- BUSINV - Total Business Inventories - Millions of Dollars\n- ALTSALES - Light Weight Vehicle Sales - Millions of Units\n- UMCSENT - Univ of Michigan: Consumer Sentiment - Index 1966:Q1=100\n- STLFSI - St. Louis Financial Stress Index - Index\n- WTISPLC - Spot Oil Price - West Texas Intermediate - Dollars per Barrel\n- CPIAUCSL - Consumer Price Index: Seasonally Adj. - Index 1982-1984=100\n- UNRATE - Civilian Total Unemployment Rate - Percent\n- UEMP27OV - Long Term Unemployment: >27 WKS - Thousands of Persons\n- UEMPMED - Length of Unemployment - Weeks\n- CE16OV - Total US Workforce - Thousands of Persons\n- EMRATIO - US Employment/Population Ratio - Percent\n- POP - US Population - Thousands\n- AHEMAN - Avg Hourly Earnings: Manufacturing - Dollars per Hour\n- AWHMAN - Avg Weekly Hours: Manufacturing - Hours\n- AWOTMAN - Avg Weekly OT Hours: Manufacturing - Hours\n- DEXUSUK - USD/GBP Currency Exchange Rate - U.S. Dollars to One U.K. Pound Sterling\n- DEXUSEU - USD/EUR Currency Exchange Rate - U.S. Dollars to One Euro\n- DEXJPUS - JPN/USD Currency Exchange Rate - Japanese Yen to One U.S. Dollar\n- DEXMXUS - MXP/USD Currency Exchange Rate - Mexican Pesos to One U.S. Dollar\n- DEXCAUS - CAD/USD Currency Exchange Rate - Canadian Dollars to One U.S. Dollar\n- DEXCHUS - CNY/USD Currency Exchange Rate - Chinese Yuan Renminbi to One U.S. Dollar\n- COMPOUT - Commercial Paper Outstanding - Billions of Dollars\n- VIXCLS - CBOE Volatility Index - Index\n- GDP - US Gross Domestic Product - Billions of Dollars\n- GNP - US Gross National Product - Billions of Dollars\n- GDI - US Gross Domestic Income - Billions of Dollars\n- NETFI - US Current Account Balance - Billions of Dollars\n- EXPGS - US Exports Goods & Services - Billions of Dollars\n- IMPGS - US Imports Goods & Services - Billions of Dollars\n- DGI - Fed Govt: Defense Budget - Billions of Dollars\n- FGRECPT - Fed Govt: Tax Receipts - Billions of Dollars\n- TGDEF - Fed Govt: Budget Deficit - Billions of Dollars\n- CP - Corporate Profits After Tax - Billions of Dollars\n- DIVIDEND - Corporate Dividends - Billions of Dollars\n- PI - Personal Income - Billions of Dollars\n- PSAVE - Personal Savings - Billions of Dollars\n- PCE - Personal Consumption Expenditures - Billions of Dollars\n- PSAVERT - Personal Savings Rate - Percent\n- MORTGAGE30US - 30-yr Conventional Mortgage Rate - Percent\n- DPCREDIT - Discount Rate - Percent\n- FEDFUNDS - Effective Federal Funds Rate - Percent\n- M1 - M1 Money Supply - Billions of Dollars\n- M2 - M2 Money Supply - Billions of Dollars\n- M1V - Velocity of M1 Money Stock - Ratio\n- M2V - Velocity of M2 Money Stock - Ratio\n- PPIACO - Producer Price Index: All Commodities - Index 1982=100\n- IMPCH - Imports from China - Millions of Dollars\n- IMPJP - Imports from Japan - Millions of Dollars\n- IMPMX - Imports from Mexico - Millions of Dollars\n- IMPCA - Imports from Canada - Millions of Dollars\n- IMPGE - Imports from Germany - Millions of Dollars\n- IMPUK - Imports from UK - Millions of Dollars\n- EXPCH - Exports to China - Millions of Dollars\n- EXPJP - Exports to Japan - Millions of Dollars\n- EXPMX - Exports to Mexico - Millions of Dollars\n- EXPCA - Exports to Canada - Millions of Dollars\n- EXPGE - Exports to Germany - Millions of Dollars\n- EXPUK - Exports to UK - Millions of Dollars\n- BOPGEXP - Exports: Goods - Millions of Dollars\n- BOPGIMP - Imports: Goods - Millions of Dollars\n- BOPGTB - Balance: Goods - Millions of Dollars\n- BOPSIMP - Imports: Services - Millions of Dollars\n- BOPSTB - Balance: Services - Millions of Dollars\n- BOPGSTB - Balance: Goods & Services - Millions of Dollars\nNote: (M)=Monthly, (Q)=Quarterly, (W)=Weekly, (D)=Daily, (Y)=Yearly\n\nWhen asked about economic data, use the get_fred_data function with the appropriate series_id.\n\nYou will receive data from ALL tool calls.\n\nIMPORTANT: \n1. Always specify start_date and end_date based on the user's question. Always use YYYY-MM-DD format, NOT relative dates like \"-2y\"\n2. If no time period specified, use recent 1 year by default\n3. Each series_id should only be called ONCE with a single continuous date range.\n   - WRONG: calling GDP twice with 2018-2020 and 2021-2023\n   - RIGHT: calling GDP once with 2018-2023\n4. Today is 2026-02-25\n"
#       },
#       {
#         "role": "user",
#         "content": "How has consumer sentiment and saving behavior changed since 2024?"
#       }]
# inputs = tokenizer.apply_chat_template(
#     messages,
#     # tools = TOOLS,
#     tokenize = True,
#     add_generation_prompt = True, # Must add for generation
#     return_tensors = "pt",
# ).to("cuda")

# from transformers import TextStreamer
# text_streamer = TextStreamer(tokenizer, skip_prompt = True)
# _ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
#                    use_cache = True, temperature = 1.5, min_p = 0.1)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("llama_lora")  # Local saving
tokenizer.save_pretrained("llama_lora")
# model.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "llama_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
      {
        "role": "system",
        "content": "You are an economic data assistant with access to FRED API. Available FRED Economic Indicators:\n\nFormat: - SERIES_ID - Indicator Name - Unit\n- TREAST - US Treasuries Held by the Fed - Billions of Dollars\n- WSHOMCB - Mortgage Backed Sec Held by the Fed - Millions of Dollars\n- WALCL - All Fed Reserve Banks - Total Assets - Millions of Dollars\n- TOTBKCR - Bank Credit of All Commercial Banks - Billions of Dollars\n- TOTALSEC - Securitized Total Consumer Loans - Millions of Dollars\n- TOTALSL - Total Consumer Credit Outstanding - Millions of Dollars\n- INVEST - Total Investments All Commercial Banks - Millions of Dollars\n- USGSEC - US Treasury and Agency Securities at All Commercial Banks - Billions of Dollars\n- CONSUMER - Total Consumer Loans - Millions of Dollars\n- BUSLOANS - Total Commercial/Industrial Loans - Billions of Dollars\n- DALLCACBEP - Delinquencies On All Loans And Leases - Billions of Dollars\n- T10Y2Y - US 10-YR / 2-YR Spread - Percent\n- TB3MS - 3-Month T-Bill: Secondary Market Rate - Percent\n- DGS10 - 10-Yr Treasury Const. Maturity Rate - Percent\n- GFDEBTN - Federal Government Debt (Public) - Millions of Dollars\n- FYOINT - Interest on National Debt - Millions of Dollars\n- FYONET - Federal Spending - Millions of Dollars\n- FYFR - Federal Receipts - Millions of Dollars\n- FYFSD - Budget Deficit/Surplus - Millions of Dollars\n- CDSP - Consumer Debt/Income Ratio - Percent\n- PERMIT - New Home Permits - Thousands of Units\n- HSN1F - New Home Sales - Thousands\n- CMDEBT - Outstanding Mortgage Debt - Millions of Dollars\n- DGORDER - Manufacturers' New Orders - Millions of Dollars\n- TCU - Capacity Utilization: Total Industry - Percent\n- TTLCONS - Total Construction Spending - Millions of Dollars\n- BUSINV - Total Business Inventories - Millions of Dollars\n- ALTSALES - Light Weight Vehicle Sales - Millions of Units\n- UMCSENT - Univ of Michigan: Consumer Sentiment - Index 1966:Q1=100\n- STLFSI - St. Louis Financial Stress Index - Index\n- WTISPLC - Spot Oil Price - West Texas Intermediate - Dollars per Barrel\n- CPIAUCSL - Consumer Price Index: Seasonally Adj. - Index 1982-1984=100\n- UNRATE - Civilian Total Unemployment Rate - Percent\n- UEMP27OV - Long Term Unemployment: >27 WKS - Thousands of Persons\n- UEMPMED - Length of Unemployment - Weeks\n- CE16OV - Total US Workforce - Thousands of Persons\n- EMRATIO - US Employment/Population Ratio - Percent\n- POP - US Population - Thousands\n- AHEMAN - Avg Hourly Earnings: Manufacturing - Dollars per Hour\n- AWHMAN - Avg Weekly Hours: Manufacturing - Hours\n- AWOTMAN - Avg Weekly OT Hours: Manufacturing - Hours\n- DEXUSUK - USD/GBP Currency Exchange Rate - U.S. Dollars to One U.K. Pound Sterling\n- DEXUSEU - USD/EUR Currency Exchange Rate - U.S. Dollars to One Euro\n- DEXJPUS - JPN/USD Currency Exchange Rate - Japanese Yen to One U.S. Dollar\n- DEXMXUS - MXP/USD Currency Exchange Rate - Mexican Pesos to One U.S. Dollar\n- DEXCAUS - CAD/USD Currency Exchange Rate - Canadian Dollars to One U.S. Dollar\n- DEXCHUS - CNY/USD Currency Exchange Rate - Chinese Yuan Renminbi to One U.S. Dollar\n- COMPOUT - Commercial Paper Outstanding - Billions of Dollars\n- VIXCLS - CBOE Volatility Index - Index\n- GDP - US Gross Domestic Product - Billions of Dollars\n- GNP - US Gross National Product - Billions of Dollars\n- GDI - US Gross Domestic Income - Billions of Dollars\n- NETFI - US Current Account Balance - Billions of Dollars\n- EXPGS - US Exports Goods & Services - Billions of Dollars\n- IMPGS - US Imports Goods & Services - Billions of Dollars\n- DGI - Fed Govt: Defense Budget - Billions of Dollars\n- FGRECPT - Fed Govt: Tax Receipts - Billions of Dollars\n- TGDEF - Fed Govt: Budget Deficit - Billions of Dollars\n- CP - Corporate Profits After Tax - Billions of Dollars\n- DIVIDEND - Corporate Dividends - Billions of Dollars\n- PI - Personal Income - Billions of Dollars\n- PSAVE - Personal Savings - Billions of Dollars\n- PCE - Personal Consumption Expenditures - Billions of Dollars\n- PSAVERT - Personal Savings Rate - Percent\n- MORTGAGE30US - 30-yr Conventional Mortgage Rate - Percent\n- DPCREDIT - Discount Rate - Percent\n- FEDFUNDS - Effective Federal Funds Rate - Percent\n- M1 - M1 Money Supply - Billions of Dollars\n- M2 - M2 Money Supply - Billions of Dollars\n- M1V - Velocity of M1 Money Stock - Ratio\n- M2V - Velocity of M2 Money Stock - Ratio\n- PPIACO - Producer Price Index: All Commodities - Index 1982=100\n- IMPCH - Imports from China - Millions of Dollars\n- IMPJP - Imports from Japan - Millions of Dollars\n- IMPMX - Imports from Mexico - Millions of Dollars\n- IMPCA - Imports from Canada - Millions of Dollars\n- IMPGE - Imports from Germany - Millions of Dollars\n- IMPUK - Imports from UK - Millions of Dollars\n- EXPCH - Exports to China - Millions of Dollars\n- EXPJP - Exports to Japan - Millions of Dollars\n- EXPMX - Exports to Mexico - Millions of Dollars\n- EXPCA - Exports to Canada - Millions of Dollars\n- EXPGE - Exports to Germany - Millions of Dollars\n- EXPUK - Exports to UK - Millions of Dollars\n- BOPGEXP - Exports: Goods - Millions of Dollars\n- BOPGIMP - Imports: Goods - Millions of Dollars\n- BOPGTB - Balance: Goods - Millions of Dollars\n- BOPSIMP - Imports: Services - Millions of Dollars\n- BOPSTB - Balance: Services - Millions of Dollars\n- BOPGSTB - Balance: Goods & Services - Millions of Dollars\nNote: (M)=Monthly, (Q)=Quarterly, (W)=Weekly, (D)=Daily, (Y)=Yearly\n\nWhen asked about economic data, use the get_fred_data function with the appropriate series_id.\n\nYou will receive data from ALL tool calls.\n\nIMPORTANT: \n1. Always specify start_date and end_date based on the user's question. Always use YYYY-MM-DD format, NOT relative dates like \"-2y\"\n2. If no time period specified, use recent 1 year by default\n3. Each series_id should only be called ONCE with a single continuous date range.\n   - WRONG: calling GDP twice with 2018-2020 and 2021-2023\n   - RIGHT: calling GDP once with 2018-2023\n4. Today is 2026-02-25\n"
      },
      {
        "role": "user",
        "content": "How has consumer sentiment and saving behavior changed since 2024?"
      }]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

You can also use Hugging Face's `AutoPeftModelForCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "llama_lora", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("llama_lora")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("llama_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/llama_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("llama_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/llama_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("llama_lora")
    tokenizer.save_pretrained("llama_lora")
if False:
    model.push_to_hub("HF_USERNAME/llama_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/llama_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("llama_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if True: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/llama_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN", # Get a token at https://huggingface.co/settings/tokens
    )

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import shutil
import os

source_path = "llama_finetune_gguf"
destination_path = "/content/drive/MyDrive/llama_finetune_gguf"

# Create the destination directory if it doesn't exist
if not os.path.exists(os.path.dirname(destination_path)):
    os.makedirs(os.path.dirname(destination_path))

# Copy the directory
shutil.copytree(source_path, destination_path, dirs_exist_ok=True)

print(f"'{source_path}' copied to '{destination_path}' successfully.")
